# \(B^\pm\to K^\pm\pi^+\pi^-\) CP-violation closure benchmark

This notebook replaces the generic CP toy with a physics benchmark inspired by the 2026 LHCb amplitude analysis of
\(B^+\to K^+\pi^+\pi^-\) (arXiv:2608.12612, 2608.12613 and 2608.12614).

LHCb parameterises every non-S-wave coefficient as
\[
a_j^\pm=(x_j\pm\Delta x_j)+i(y_j\pm\Delta y_j).
\]
That is exactly the convention implemented by `CPRealImag`.

For this first closure we deliberately use a **truncated non-S-wave submodel**:
- \(K^*(892)^0\) — fixed reference;
- \(\rho(770)^0\);
- \(K_2^*(1430)^0\);
- \(f_2(1270)\);
- \(\rho_3(1690)^0\).

The input couplings are reconstructed from the published Isobar-model fit fractions, CP asymmetries, average phases and
charge-dependent phase differences. The full LHCb fit also contains many additional resonances and large \(K\pi\) and
\(\pi\pi\) S-waves; therefore this notebook is a closure benchmark for the fitter, **not** a numerical reproduction of the
complete experimental likelihood.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    CPRealImag,
    DecayChannel,
    DecayModel,
    GounarisSakurai,
    Minimizer,
    Parameter,
    RealImag,
    Resonance,
    enable_x64,
    weighted_resample,
)
from dalitzplotfitter.likelihood import SimultaneousNLL

enable_x64()


## 1. Published Isobar inputs

The following central values come from the LHCb Isobar solution:

- fit fractions and average phases: arXiv:2608.12613 / 2608.12612;
- quasi-two-body \(A_{CP}\) and \(\delta^- - \delta^+\): arXiv:2608.12614.

The reference \(K^*(892)^0\) is fixed here to \(1+0i\) for both charges. This removes the two independent overall
phase/scale gauges of the separately normalised \(B^+\) and \(B^-\) PDFs. We therefore do **not** try to reproduce the
small published \(A_{CP}(K^*(892)^0)\) in this first closure.


In [ ]:
FREF = 12.167  # % ; K*(892)^0 Isobar fit fraction

published = {
    # name: fit_fraction[%], A_CP, average phase[deg], phase(B-) - phase(B+)[deg]
    "rho770":      (7.834, +0.280,  -23.0, -40.3),
    "K2star1430":  (2.253, -0.084,  +91.2, -10.2),
    "f2_1270":     (2.152, -0.354, +150.8, +31.8),
    "rho3_1690":   (0.408, +0.603,  -96.9, -45.1),
}

def coefficients_from_observables(fit_fraction, acp, average_phase_deg, phase_difference_deg):
    # With a unit-normalised dynamical basis, the charge-specific numerator
    # fractions are proportional to |c_+|^2 and |c_-|^2.
    r_plus  = np.sqrt(fit_fraction * (1.0 - acp) / FREF)
    r_minus = np.sqrt(fit_fraction * (1.0 + acp) / FREF)

    dphi = np.deg2rad(phase_difference_deg)
    phi_avg = np.deg2rad(average_phase_deg)

    # LHCb defines the average phase as arg[(c_+ + c_-)/2], not simply
    # (phi_+ + phi_-)/2. Solve for phi_+ exactly, then phi_- = phi_+ + dphi.
    offset = np.angle(r_plus + r_minus * np.exp(1j*dphi))
    phi_plus = phi_avg - offset
    phi_minus = phi_plus + dphi

    c_plus = r_plus * np.exp(1j*phi_plus)
    c_minus = r_minus * np.exp(1j*phi_minus)

    c_average = 0.5*(c_plus + c_minus)
    c_difference = 0.5*(c_plus - c_minus)
    return dict(
        c_plus=c_plus,
        c_minus=c_minus,
        x=c_average.real,
        y=c_average.imag,
        dx=c_difference.real,
        dy=c_difference.imag,
    )

truth_coefficients = {
    name: coefficients_from_observables(*values)
    for name, values in published.items()
}

for name, values in truth_coefficients.items():
    cp, cm = values["c_plus"], values["c_minus"]
    acp = (abs(cm)**2 - abs(cp)**2)/(abs(cm)**2 + abs(cp)**2)
    dphi = np.rad2deg(np.angle(cm/cp))
    print(
        f"{name:12s} c+={cp.real:+.4f}{cp.imag:+.4f}i  "
        f"c-={cm.real:+.4f}{cm.imag:+.4f}i  "
        f"Acp={acp:+.3f}  dphi={dphi:+.1f} deg"
    )


## 2. Shared fit parameters and charge models

All four Cartesian parameters \((x,y,\Delta x,\Delta y)\) float for each non-reference component. The same `Parameter`
objects are shared by the \(B^+\) and \(B^-\) models; only `charge=+1` or `charge=-1` changes.

The LHCb analysis uses \(r_P=r_D=4\,\mathrm{GeV}^{-1}\). The \(\rho(770)^0\) is described with Gounaris–Sakurai,
while the other selected states use the relativistic Breit–Wigner implementation.


In [ ]:
cp_parameters = {}

for name, t in truth_coefficients.items():
    cp_parameters[name] = CPRealImag(
        Parameter.coefficient(f"{name}.x",  t["x"],  owner=name, bounds=(-3.0, 3.0), step=0.02),
        Parameter.coefficient(f"{name}.y",  t["y"],  owner=name, bounds=(-3.0, 3.0), step=0.02),
        Parameter.coefficient(f"{name}.dx", t["dx"], owner=name, bounds=(-1.5, 1.5), step=0.01),
        Parameter.coefficient(f"{name}.dy", t["dy"], owner=name, bounds=(-1.5, 1.5), step=0.01),
    )

channel_plus = DecayChannel("B+", ("K+", "pi+", "pi-"))
channel_minus = DecayChannel("B-", ("K-", "pi-", "pi+"))

def build_model(channel, charge):
    return DecayModel(
        channel,
        [
            Resonance(
                "Kstar892", (0, 2), RealImag(1.0, 0.0),
                mass=0.89555, width=0.0473, spin=1,
                resonance_radius=4.0, parent_radius=4.0,
            ),
            Resonance(
                "rho770", (1, 2), cp_parameters["rho770"].for_charge(charge),
                lineshape=GounarisSakurai(),
                mass=0.77526, width=0.1491, spin=1,
                resonance_radius=4.0, parent_radius=4.0,
            ),
            Resonance(
                "K2star1430", (0, 2), cp_parameters["K2star1430"].for_charge(charge),
                mass=1.4324, width=0.109, spin=2,
                resonance_radius=4.0, parent_radius=4.0,
            ),
            Resonance(
                "f2_1270", (1, 2), cp_parameters["f2_1270"].for_charge(charge),
                mass=1.2755, width=0.1867, spin=2,
                resonance_radius=4.0, parent_radius=4.0,
            ),
            Resonance(
                "rho3_1690", (1, 2), cp_parameters["rho3_1690"].for_charge(charge),
                mass=1.6888, width=0.161, spin=3,
                resonance_radius=4.0, parent_radius=4.0,
            ),
        ],
        normalization_resolution=320,
    )

model_plus = build_model(channel_plus, +1)
model_minus = build_model(channel_minus, -1)

assert tuple(p.name for p in model_plus.parameters) == tuple(p.name for p in model_minus.parameters)
truth = {p.name: float(p.value) for p in model_plus.parameters}
print("free parameters:", len(model_plus.parameters))


## 3. Generate \(B^+\) and \(B^-\) toys

The proposal samples are independent. Events are resampled with
\[
w_{\rm target}=w_{\rm PS}|A^\pm|^2.
\]
The default sizes below are intended to be large enough for a meaningful 16-parameter closure while remaining practical
on a laptop/GPU. Increase them for precision studies.


In [ ]:
N_POOL = 350_000
N_TOY = 40_000

pool_plus = model_plus.generate_phase_space(N_POOL, seed=1261201)
pool_minus = model_minus.generate_phase_space(N_POOL, seed=1261202)

target_plus = pool_plus.weights * model_plus.intensity(pool_plus.as_dict(), truth)
target_minus = pool_minus.weights * model_minus.intensity(pool_minus.as_dict(), truth)

toy_plus = weighted_resample(jax.random.key(1261203), pool_plus, target_plus, N_TOY, replace=True)
toy_minus = weighted_resample(jax.random.key(1261204), pool_minus, target_minus, N_TOY, replace=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
h0 = axes[0].hist2d(np.asarray(toy_plus.s12), np.asarray(toy_plus.s13), bins=85)
h1 = axes[1].hist2d(np.asarray(toy_minus.s12), np.asarray(toy_minus.s13), bins=85)
axes[0].set(title=r"$B^+$ toy", xlabel=r"$m^2(K^+\pi^-)$", ylabel=r"$m^2(\pi^+\pi^-)$")
axes[1].set(title=r"$B^-$ toy", xlabel=r"$m^2(K^-\pi^+)$", ylabel=r"$m^2(\pi^-\pi^+)$")
fig.colorbar(h0[3], ax=axes[0], label="events")
fig.colorbar(h1[3], ax=axes[1], label="events")
plt.show()


## 4. Visual CP asymmetry in projections


In [ ]:
bins_pipi = np.linspace(
    min(float(np.min(toy_plus.s13)), float(np.min(toy_minus.s13))),
    max(float(np.max(toy_plus.s13)), float(np.max(toy_minus.s13))),
    90,
)
hp, edges = np.histogram(np.asarray(toy_plus.s13), bins=bins_pipi)
hm, _ = np.histogram(np.asarray(toy_minus.s13), bins=bins_pipi)
centers = 0.5*(edges[:-1] + edges[1:])
asym = (hm - hp) / np.maximum(hm + hp, 1)

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True, constrained_layout=True)
axes[0].step(centers, hp, where="mid", label=r"$B^+$")
axes[0].step(centers, hm, where="mid", label=r"$B^-$")
axes[0].set(ylabel="events")
axes[0].legend()
axes[1].axhline(0.0, lw=0.8)
axes[1].step(centers, asym, where="mid")
axes[1].set(xlabel=r"$m^2(\pi^+\pi^-)$ [GeV$^2$]", ylabel=r"$(N_- - N_+)/(N_- + N_+)$")
plt.show()


## 5. Cached simultaneous likelihood

Only coefficient parameters float, so all five dynamical columns and the normalization matrices are cached once.
The fit therefore varies \(16\) Cartesian parameters without reevaluating any resonance lineshape on the data or the
normalization grid.


In [ ]:
cache_plus = model_plus.prepare_cache(toy_plus)
cache_minus = model_minus.prepare_cache(toy_minus)

def cached_nll(cache):
    def term(values):
        intensity, normalization = cache.evaluate(values)
        tiny = jnp.finfo(intensity.dtype).tiny
        return (
            -jnp.sum(jnp.log(jnp.maximum(intensity, tiny)))
            + intensity.shape[0]*jnp.log(normalization)
        )
    return term

objective = SimultaneousNLL((
    cached_nll(cache_plus),
    cached_nll(cache_minus),
))

parameters = model_plus.parameters
fitter = Minimizer(objective, parameters, tolerance=1e-5, verbose=1)


## 6. One randomized start and one simultaneous fit

This is a genuine closure test: the truth generates the events, all 16 floating parameters are randomized once, and a
single MIGRAD/HESSE fit is run. No multistart scan is used.


In [ ]:
start = fitter.random_start(seed=20260830)

print(f"{'parameter':18s} {'truth':>10s} {'start':>10s}")
for p in parameters:
    print(f"{p.name:18s} {truth[p.name]:10.4f} {start[p.name]:10.4f}")

result = fitter.fit(start_values=start, simplex=False, ncall=30000)
print(result.fmin)


## 7. Parameter closure


In [ ]:
names = [p.name for p in parameters if not p.fixed]
fit_values = {name: float(result.values[name]) for name in names}
fit_errors = {name: float(result.errors[name]) for name in names}

print(f"{'parameter':18s} {'truth':>10s} {'start':>10s} {'fit':>10s} {'error':>10s} {'pull':>9s}")
pulls = []
for name in names:
    pull = (fit_values[name] - truth[name]) / fit_errors[name]
    pulls.append(pull)
    print(
        f"{name:18s} {truth[name]:10.4f} {start[name]:10.4f} "
        f"{fit_values[name]:10.4f} {fit_errors[name]:10.4f} {pull:9.3f}"
    )

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0.0, lw=0.8)
ax.axhline(+1.0, lw=0.6, ls="--")
ax.axhline(-1.0, lw=0.6, ls="--")
ax.scatter(np.arange(len(names)), pulls)
ax.set_xticks(np.arange(len(names)), names, rotation=70, ha="right")
ax.set(ylabel="pull", title="CP coefficient closure")
plt.tight_layout()
plt.show()


## 8. Recover the physical CP observables

From the fitted Cartesian coefficients we reconstruct
\[
A_{CP}^j=\frac{|c_j^-|^2-|c_j^+|^2}{|c_j^-|^2+|c_j^+|^2},
\qquad
\Delta\phi_j=\arg(c_j^-)-\arg(c_j^+).
\]
These are the observables used to construct the truth model.


In [ ]:
def physical_observables(coefficient, values):
    cp = complex(coefficient.for_charge(+1).value(values))
    cm = complex(coefficient.for_charge(-1).value(values))
    acp = (abs(cm)**2 - abs(cp)**2)/(abs(cm)**2 + abs(cp)**2)
    dphi = np.rad2deg(np.angle(cm/cp))
    return cp, cm, acp, dphi

rows = []
for name, coefficient in cp_parameters.items():
    _, _, acp_t, dphi_t = physical_observables(coefficient, truth)
    _, _, acp_f, dphi_f = physical_observables(coefficient, fit_values)
    rows.append((name, acp_t, acp_f, dphi_t, dphi_f))

print(f"{'component':12s} {'ACP truth':>11s} {'ACP fit':>11s} {'dphi truth':>12s} {'dphi fit':>12s}")
for row in rows:
    print(f"{row[0]:12s} {row[1]:11.3f} {row[2]:11.3f} {row[3]:12.2f} {row[4]:12.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
labels = [r[0] for r in rows]
xpos = np.arange(len(rows))
axes[0].scatter(xpos, [r[1] for r in rows], marker="x", s=90, label="truth")
axes[0].scatter(xpos, [r[2] for r in rows], marker="o", s=50, label="fit")
axes[0].axhline(0, lw=0.7)
axes[0].set_xticks(xpos, labels, rotation=30, ha="right")
axes[0].set(ylabel=r"$A_{CP}$", title="Quasi-two-body CP asymmetry")
axes[0].legend()

axes[1].scatter(xpos, [r[3] for r in rows], marker="x", s=90, label="truth")
axes[1].scatter(xpos, [r[4] for r in rows], marker="o", s=50, label="fit")
axes[1].axhline(0, lw=0.7)
axes[1].set_xticks(xpos, labels, rotation=30, ha="right")
axes[1].set(ylabel=r"$\delta^- - \delta^+$ [deg]", title="CP-violating phase difference")
axes[1].legend()
plt.show()


## Interpretation

A successful run should recover, within statistical fluctuations, the large published-inspired CP effects in the
\(\rho(770)^0\), \(f_2(1270)\) and \(\rho_3(1690)^0\) coefficients while remaining compatible with the smaller
\(K_2^*(1430)^0\) effect.

The next refinement is to add the dominant S-wave sector (GLASS/K-matrix/QMI) and the remaining nonzero-spin states from
the complete LHCb model. That step should only be attempted after this non-S-wave closure is stable, because the paper
shows that the S-wave treatment is the main model-dependence in this channel.
